# Summer Curriculum - Week 1

## Build the nn

In [2]:
# Import necessary libraries
import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

next I am going to select the device that we are going to be building the nn on

In [4]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using mps device


Next we are going to subclass the nn module so that we get all of the already-built in features that we want from it before customizing it for our purposes

In [11]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )
        
    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

Next we are going to make an instance of the nn and then move it to the desired device that we selected.

In [12]:
model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In order to use the model, we pass it the input data. this will execute the model's forard movement along with a few background operations. So we should not call model.forward()

In [13]:
X = torch.rand(1, 28, 28, device = device)
logits = model(X)
pred_probab = nn.Softmax(dim=1)(logits)
y_pred = pred_probab.argmax(1)
print(f"Predicted class: {y_pred}")

Predicted class: tensor([6], device='mps:0')


Next we are going to break down the layers in the model using a minibatch.

In [14]:
input_image = torch.rand(3,28,28)
print(input_image.size())

torch.Size([3, 28, 28])


We will next initialize the 'nn.Flatten' layer to convert each 2D 28x28 image into a contiguous array of 784 pixel values (the minibatch dimension (at dim = 0) is maintained).

In [15]:
flatten = nn.Flatten()
flat_image = flatten(input_image)
print(flat_image.size())

torch.Size([3, 784])


the linear layer is a module that applies a linear transformation on the input using its stored weights and biases.

In [16]:
layer1 = nn.Linear(in_features = 28*28, out_features = 20)
hidden1 = layer1(flat_image)
print(hidden1.size())

torch.Size([3, 20])


Non-linear activations are what create the complex mappings between the model's inputs and outputs. ey are applied after linear transformations to introduce nonlinearity, helping neural networks learn a wide variety of phenomena.

In this model, we use 'nn.ReLU' between our linear layers but there are other activations that can introduce non-linearity in our model.

In [17]:
print(f"Before ReLU: {hidden1}\n\n")
hidden1 = nn.ReLU()(hidden1)
print(f"After ReLU: {hidden1}")

Before ReLU: tensor([[ 0.0359,  0.9991, -0.0705,  0.1159,  0.2290,  0.2291,  0.1875, -0.2584,
          0.1502,  0.2713, -0.1915,  0.0151, -0.0660,  0.0211,  0.0817,  0.1939,
          0.5994,  0.2407, -0.4638,  0.0508],
        [ 0.1780,  0.8954,  0.3556,  0.0896,  0.1601, -0.0491,  0.5279, -0.1160,
         -0.3497,  0.4333, -0.1873,  0.2301, -0.3414, -0.3118, -0.0044, -0.2953,
          0.4459,  0.0577,  0.0050, -0.2028],
        [ 0.3116,  0.5667,  0.1190,  0.2762,  0.1459,  0.0838,  0.4578,  0.0848,
          0.1169,  0.4450, -0.1168, -0.0546, -0.1299, -0.1928, -0.1649,  0.2783,
          0.3679, -0.0077, -0.3179, -0.2066]], grad_fn=<AddmmBackward0>)


After ReLU: tensor([[0.0359, 0.9991, 0.0000, 0.1159, 0.2290, 0.2291, 0.1875, 0.0000, 0.1502,
         0.2713, 0.0000, 0.0151, 0.0000, 0.0211, 0.0817, 0.1939, 0.5994, 0.2407,
         0.0000, 0.0508],
        [0.1780, 0.8954, 0.3556, 0.0896, 0.1601, 0.0000, 0.5279, 0.0000, 0.0000,
         0.4333, 0.0000, 0.2301, 0.0000, 0.0000, 0.00

nn.Sequential is an ordered container of modules. the data is passef through all the modules in the same order as defined. You can use sequential containers to put together a quick network like 'seq_modeules'.

In [21]:
seq_modules = nn.Sequential(
    flatten,
    layer1,
    nn.ReLU(),
    nn.Linear(20, 10)
)
input_image = torch.rand(3,28,28)
logits = seq_modules(input_image)

the final layer of the neural network returns logits - raw values in the range of negative infinity to infinity - which are passed to the nn.Softmax module. the logits are scaled to values [0,1] representing the model's predicted probabilities for each class. the 'dim' parameter indicates the dimension which the values must sum to 1.

In [23]:
softmax = nn.Softmax(dim=1)
pred_probab = softmax(logits)

Many layers inside a neural network are parameterized - have associated weights and biases that are optimnized during training. Subclassing nn.Module automatically tracks all fields defined inside your model object, and makes all parameters using your model's 'parameters()' or 'named_parameters()' methods. We are going to iterate over each parameter and print its size and a preview of its values.

In [24]:
print(f"Model structure: {model}\n\n")

for name, param in model.named_parameters():
    print(f"Layer {name} | Size: {param.size()} | Values: {param[:2]} \n")

Model structure: NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


Layer linear_relu_stack.0.weight | Size: torch.Size([512, 784]) | Values: tensor([[-0.0042, -0.0127, -0.0302,  ...,  0.0043, -0.0166,  0.0134],
        [ 0.0217, -0.0045,  0.0167,  ..., -0.0064,  0.0108, -0.0291]],
       device='mps:0', grad_fn=<SliceBackward0>) 

Layer linear_relu_stack.0.bias | Size: torch.Size([512]) | Values: tensor([-0.0029, -0.0306], device='mps:0', grad_fn=<SliceBackward0>) 

Layer linear_relu_stack.2.weight | Size: torch.Size([512, 512]) | Values: tensor([[-0.0427, -0.0121, -0.0116,  ..., -0.0205,  0.0357, -0.0076],
        [-0.0194, -0.0406, -0.0240,  ...,  0.0340,  0.0056, -0.0154]],
       device='mps:0', grad_fn=<SliceBackwa